# Take-Home Case: The Mortgage Book of Aare-Säntis Regionalbank AG

**EAIF: AI for Finance — team take-home between the classes on linear/logistic regression and on advanced supervised learning**

You are the newly formed data team of Aare-Säntis Regionalbank AG, a fictional regional bank in the Swiss Mittelland. The bank has 10,000 residential mortgages on its books, originated between 2019 and 2022, spread over seven cantons. Until now, the bank's credit decisions have rested on a vendor valuation model, a handful of ratio rules and the judgement of the branch advisers. This morning the Chief Risk Officer sent the team its first assignment.

> **Memo from the CRO**
>
> To the data team. Welcome aboard. Three things, in order of urgency.
>
> 1. Our collateral valuations come from a vendor model we cannot inspect. Build one we can.
> 2. About seven in a hundred of our recent mortgages ran into payment trouble within three years. Tell me which applications carry that risk, and where we should set the approval bar, in francs.
> 3. The risk committee has read that banks use "machine learning" for this. Show me whether the more flexible methods actually do better on our book, and whether we could defend deploying one.
>
> I do not need a slide deck. I need numbers I can trust and one paragraph per question that I can read out to the committee.
>
> Head of Risk

This notebook is your working file for the assignment. Part A answers the first two requests with the methods you already know, linear and logistic regression. Part B answers the third with the methods introduced in the next class. Every section ends with what you, the analyst, tell the CRO.

## The data

Two files are loaded from the course repository. `mortgages.csv` is the bank's book: 10,000 mortgages originated 2019 to 2022, each with its outcome after 36 months. `applications_2026.csv` holds this year's 500 applications. It has the same columns as the book except two: the outcome, which does not exist yet because the bank has not decided on them, and one further column that Part A2 comes to. The applications file has 22 columns where the book has 24. The full column dictionary is in [`data/mortgage2026/README.md`](https://github.com/umatter/EDFB/blob/main/data/mortgage2026/README.md); the groups of columns are these.

| Group | Columns |
|---|---|
| Property | `canton`, `property_type`, `living_area_m2`, `rooms`, `year_built`, `distance_center_km`, `energy_label`, `purchase_price` |
| Borrower | `household_income`, `age`, `employment`, `years_client` |
| Loan | `loan_amount`, `rate_type`, `fixed_years`, `interest_rate`, `amortisation`, `origination_year` |
| Derived ratios | `ltv`, `affordability`, `actual_burden` |
| Outcome | `trouble_36m` (1 if the mortgage was 90 days or more in arrears, or was restructured, within 36 months) |

Three ratios do most of the work in Swiss mortgage lending, and all three are in the file.

The loan-to-value ratio compares the loan with the price of the property: `ltv = loan_amount / purchase_price`. Swiss banks normally finance at most 80 % of the price, and the part of the loan above two-thirds of the price must be amortised within 15 years.

The affordability ratio is the Swiss lending rule for whether the household can carry the loan through a rise in interest rates. It does not use the interest rate actually agreed but an imputed rate of 5 %, adds 1 % of the purchase price per year for maintenance, adds the amortisation of the part above two-thirds LTV spread over 15 years, and divides the sum by gross household income:

`affordability = (0.05 × loan_amount + 0.01 × purchase_price + amortisation per year) / household_income`

The rule says the ratio must not exceed one third. A household with CHF 150,000 gross income and a CHF 800,000 loan on a CHF 1,000,000 property carries 40,000 of imputed interest, 10,000 of maintenance and about 8,900 of amortisation per year, which is 0.39 of its income, above the bar.

The actual burden is what the household pays in interest today, at the agreed rate: `actual_burden = interest_rate × loan_amount / household_income`. With rates mostly between 1 and 2 %, it is a fraction of the affordability ratio, which is exactly why the imputed rate exists.

> **Simulated data.** The two files were generated by the course for this case. There is no real bank, property or household behind any row. The rate of payment trouble in the book is several times higher than in a real Swiss mortgage book so that the models have enough cases to learn from; the ratios, prices and incomes are in a realistic range but were not taken from any real data source.

One more thing to know before you start: the book contains one column that the bank does *not* know at the moment it decides on an application. Part A2 deals with it.

## How to work through this notebook

Plan three hours for a team of three to four. Make a copy of the notebook in your Drive (File, Save a copy in Drive) and work in the copy. Run the given cells one after the other and read them, including the comments; they are the worked part of the case and they are where the methods are explained. The exercises are marked `### Exercise n` and each one is followed by a code cell that holds only a comment. Fill that cell and leave the given cells as they are, because later parts of the notebook use the objects they create.

Rotate who types for each part, so that nobody sits through the whole case as a spectator. In the next class, any member of the team can be asked to explain any cell, worked or exercise, so make sure everyone can. The notebook is discussed in that class; it is not graded.

Every choice of columns in this case follows one rule, which you will meet again in every part:

**At the moment the bank decides on an application, which columns are already known?**

Everything a model uses must pass that test. A model that is fed a column the bank only learns afterwards looks excellent in the notebook and is useless at the counter.

## Roadmap

| Part | Question | Method | Exercises |
|---|---|---|---|
| A1 | Is our collateral valued right? | Linear regression | 1, 2 |
| A2 | Which applications will run into payment trouble, and where is the approval bar in francs? | Logistic regression | 3, 4 |
| B | Do the flexible methods do better on our book? | LASSO, decision tree, random forest, gradient boosting and XGBoost, SVM, comparison | |
| C | Your turn on the new methods | The methods of Part B, and reflection questions for the class | 5 to 8 |

## Setup

In [ ]:
# Setup: Colab's preinstalled stack only, nothing to install
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance, DecisionBoundaryDisplay
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score, roc_auc_score,
                             roc_curve, confusion_matrix, precision_score, recall_score)
import xgboost as xgb

np.random.seed(0)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
sns.set_theme(style="whitegrid")

# The two error costs the CRO gave us, in CHF, used throughout the notebook
COST_FN = 60_000   # a loan that runs into trouble was approved: expected loss
COST_FP = 8_000    # a loan that would have been fine was rejected: lost margin

print("Setup complete. scikit-learn", __import__("sklearn").__version__, "| xgboost", xgb.__version__)

In [ ]:
# Load the two data files from the course repository
DATA = "https://raw.githubusercontent.com/umatter/EDFB/main/data/mortgage2026/"
book = pd.read_csv(DATA + "mortgages.csv")
apps = pd.read_csv(DATA + "applications_2026.csv")
print("book:", book.shape, "| applications:", apps.shape)
book.head()

In [ ]:
# Types, missing values, and the outcome's base rate
print(book.dtypes, "\n")
print("missing values:", int(book.isna().sum().sum()))
print(f"trouble rate in the book: {book.trouble_36m.mean():.3%}  ({book.trouble_36m.sum()} of {len(book)})")
book.describe().T

# Part A: Recap

## A1. Is our collateral valued right? Linear regression

The bank lends against the property. If the borrower stops paying, the bank sells the property and recovers what it can, so the question behind every mortgage is what the property is worth, as opposed to what the buyer paid for it. Today that question is answered by a vendor model that returns a number and no explanation. The CRO's first request is a valuation the bank can inspect.

The standard tool for this is a hedonic model: the price of a property is written as the sum of the prices of its characteristics. A linear regression is exactly such a model, and it is inspectable by construction. Its coefficients are a price per square metre, a premium or discount per canton, a discount per kilometre from the regional centre, a premium for a house over an apartment. A valuer can read those numbers, argue with them, and compare them with what the market pays.

We follow the steps of the first supervised-learning class.

1. Look at the data, in a plot, before fitting anything.
2. Pick the target Y (`purchase_price`) and the features X (the property columns).
3. Split the book into a training set and a test set.
4. Fit the model on the training set.
5. Test it on the held-out data, and put its RMSE next to the RMSE of a baseline that knows nothing.

The features are all property characteristics, which the bank knows when the application arrives, so the decision-time rule is satisfied.

In [ ]:
# Step 1: look at the relationship we want to model. Price against living area, one point per mortgage.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(book.living_area_m2, book.purchase_price / 1e6, s=6, alpha=0.3)
ax.set_xlabel("living area (m²)")
ax.set_ylabel("purchase price (CHF million)")
ax.set_title("The bank's book: purchase price against living area")
plt.show()

In [ ]:
# Step 2: the simplest model. One X, one slope: CHF per square metre, averaged over everything else.
uni = LinearRegression().fit(book[["living_area_m2"]], book.purchase_price)
print(f"price = {uni.intercept_:,.0f} + {uni.coef_[0]:,.0f} x living area")
print(f"R² on the full book: {uni.score(book[['living_area_m2']], book.purchase_price):.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(book.living_area_m2, book.purchase_price / 1e6, s=6, alpha=0.3)
grid = np.linspace(45, 320, 50).reshape(-1, 1)
ax.plot(grid, uni.predict(pd.DataFrame(grid, columns=["living_area_m2"])) / 1e6, color="C3", lw=2)
ax.set_xlabel("living area (m²)"); ax.set_ylabel("purchase price (CHF million)")
ax.set_title("Simple linear regression: one slope for all cantons")
plt.show()

The slope is the whole model. In this run it says that one additional square metre of living area adds CHF 7,589 to the price, averaged over every canton, every building age and every location in the book, and living area alone explains 48 % of the variation in prices (an R² of 0.484). The line runs through the middle of the cloud, but the cloud is wide: at 150 m² the book holds properties that sold for well under one million and others that sold for close to two.

One slope is not enough because a square metre does not cost the same everywhere. A square metre in the canton of Zurich and one in the canton of Solothurn are priced in different markets. A house built in 1960 and one built in 2018 differ in what a buyer pays, as do a flat next to the station and one twenty kilometres out. The simple model averages over all of that, and the spread around the line is the price of averaging.

The multivariate model puts these characteristics in as further columns of X. The numeric ones (`living_area_m2`, `year_built`, `distance_center_km`) enter as they are. The categorical ones (`canton`, `property_type`, `energy_label`) become 0/1 dummy columns, one per level, with one reference level dropped per column, exactly as in the logistic-regression class: the coefficient of each dummy is then the premium relative to the dropped level. `pd.get_dummies(..., drop_first=True)` drops the alphabetically first level, so the references are canton AG, property type `apartment` and energy label A.

In [ ]:
# Step 3: the multivariate model. Categorical columns become 0/1 dummies (one reference level dropped each).
price_features = ["canton", "property_type", "living_area_m2", "year_built", "distance_center_km", "energy_label"]
Xp = pd.get_dummies(book[price_features], drop_first=True).astype(float)
yp = book.purchase_price

Xp_train, Xp_test, yp_train, yp_test = train_test_split(Xp, yp, test_size=0.3, random_state=42, stratify=book.canton)
lin = LinearRegression().fit(Xp_train, yp_train)

coef = pd.Series(lin.coef_, index=Xp.columns).sort_values()
print("intercept:", f"{lin.intercept_:,.0f}")
coef.round(0).to_frame("CHF per unit")

Every row of the table is a price the bank can read. In this run, one square metre of living area adds CHF 6,705, now holding canton, building age, location and energy label fixed, which is about CHF 900 less than the slope of the simple model. The canton dummies are premiums relative to Aargau, the dropped reference: a property in the canton of Zurich sells for about CHF 403,000 more than the same property in Aargau, one in Solothurn for about CHF 187,000 less. Each kilometre further from the regional centre takes about CHF 17,900 off the price. A house sells for about CHF 96,000 more than an apartment with the same area, age, canton and location, and each energy label below A carries its own discount.

These numbers are the inspectable model the CRO asked for. A valuer who disagrees with the Zurich premium or the discount per kilometre can say so, in francs, and the bank can check the number against recent transactions.

In [ ]:
# Step 4: test the model on data it has not seen, next to the baseline "predict the training mean"
def rmse_of(y, pred):
    return float(np.sqrt(mean_squared_error(y, pred)))

pred_test = lin.predict(Xp_test)
baseline = np.full(len(yp_test), yp_train.mean())
print(f"RMSE  linear model : CHF {rmse_of(yp_test, pred_test):>10,.0f}   (R² {r2_score(yp_test, pred_test):.3f})")
print(f"RMSE  predict mean : CHF {rmse_of(yp_test, baseline):>10,.0f}   (R² {r2_score(yp_test, baseline):.3f})")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(yp_test / 1e6, pred_test / 1e6, s=6, alpha=0.3)
lim = [0, yp_test.max() / 1e6]
axes[0].plot(lim, lim, color="C3", lw=1)
axes[0].set_xlabel("actual price (CHF million)"); axes[0].set_ylabel("predicted price (CHF million)")
axes[0].set_title("Test set: predicted against actual")
axes[1].scatter(pred_test / 1e6, (yp_test - pred_test) / 1e6, s=6, alpha=0.3)
axes[1].axhline(0, color="C3", lw=1)
axes[1].set_xlabel("predicted price (CHF million)"); axes[1].set_ylabel("residual (CHF million)")
axes[1].set_title("Residuals fan out as the price grows")
plt.show()

The test set holds 3,000 mortgages the model has never seen. In this run the linear model's RMSE on the test set is CHF 154,786, against CHF 426,031 for the baseline that predicts the training mean for every property, and it explains 87 % (an R² of 0.868) of the variation in test prices where the baseline explains none. For a property that the model values at CHF 900,000, an error of one RMSE means that the price such a property fetches is plausibly anywhere between about CHF 745,000 and CHF 1,055,000, which is the uncertainty the bank has to keep in mind when it sets the loan-to-value ratio.

The right-hand plot shows something the RMSE hides. The residuals are not a band of constant width; they fan out. Cheap properties are missed by a small amount, expensive ones by a large amount, and the error grows in proportion to the price. That is the signature of a multiplicative relationship: a Zurich premium is more naturally a percentage of the price than a fixed sum in francs, and so is the discount for an old building. A model in levels cannot express that, which is why it is imprecise exactly where the bank's exposures are largest. Exercise 1 takes the hint.

What the analyst tells the CRO: a linear regression on six property characteristics values the collateral to within about CHF 155,000 on unseen properties, every coefficient is a price in francs that a valuer can inspect and challenge, and the model's main weakness, its imprecision for expensive properties, can be addressed by modelling the price in logarithms, which is the next step.

### Exercise 1: A model in logarithms

Fit the same multivariate model on `np.log(yp_train)` instead of `yp_train`. Predict on the test set, transform the predictions back with `np.exp`, and report the RMSE in CHF next to the RMSE of the model in levels. Then plot the residuals of the log model against its predictions as in the cell above.

*Deliverable:* the two RMSE values, and one sentence on which model the bank should use and why (look at the residual plot, not only at the RMSE).

In [ ]:
# Exercise 1: solution
lin_log = LinearRegression().fit(Xp_train, np.log(yp_train))
pred_log = np.exp(lin_log.predict(Xp_test))
print(f"RMSE  log model    : CHF {rmse_of(yp_test, pred_log):>10,.0f}   (R² {r2_score(yp_test, pred_log):.3f})")
print(f"RMSE  levels model : CHF {rmse_of(yp_test, pred_test):>10,.0f}   (R² {r2_score(yp_test, pred_test):.3f})")

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(pred_log / 1e6, (yp_test - pred_log) / 1e6, s=6, alpha=0.3)
ax.axhline(0, color="C3", lw=1)
ax.set_xlabel("predicted price, log model (CHF million)"); ax.set_ylabel("residual (CHF million)")
ax.set_title("Log model: residuals still grow with the price, but more evenly")
plt.show()

# The log model's coefficients are percentage effects: exp(coef) - 1
pct = (np.exp(pd.Series(lin_log.coef_, index=Xp.columns)) - 1).sort_values()
(pct * 100).round(1).to_frame("% effect on price")

**What we expect them to find.** The log model has the lower test RMSE in this run and its coefficients read as percentages (a Zurich premium of about 50 % over Aargau, a discount of about 2 % per kilometre from the centre), which is how valuers talk. Its residuals are more homogeneous in relative terms. The bank should use the log model; the levels model's fan means it is systematically too imprecise for expensive properties, which are exactly the large exposures.

### Exercise 2: Did the buyer overpay? A feature for Part B

Use your log model to predict a value for **every** property in the book (`Xp`, not only the test set) and compute `overpayment = purchase_price / predicted value`. A value above 1 means the buyer paid more than comparable properties cost. Store it as `book["overpayment"]`. Then compare the trouble rate of the mortgages in the top 10 % of `overpayment` with the rest.

*Deliverable:* the two trouble rates and one sentence on whether the valuation model tells the bank something about repayment risk. (Part B recomputes this column itself, so the notebook keeps running if you skip this.)

In [ ]:
# Exercise 2: solution
book["overpayment"] = book.purchase_price / np.exp(lin_log.predict(Xp))
top = book.overpayment >= book.overpayment.quantile(0.9)
print(book.overpayment.describe().round(3))
print(f"trouble rate, top 10% overpayment: {book.trouble_36m[top].mean():.3%}")
print(f"trouble rate, the rest           : {book.trouble_36m[~top].mean():.3%}")

**What we expect them to find.** The top decile of overpayment has a clearly higher trouble rate than the rest in this run (the generator plants a moderate effect). The story to tell the CRO: a buyer who paid more than the model value is financed at a higher loan-to-*value* than the loan-to-*price* suggests, so the valuation model is also a risk signal. Some teams will use the levels model; the ranking is nearly the same.

## A2. Which borrowers run into payment trouble? Logistic regression

The CRO's second request is different in kind from the first. The target is no longer a price but a yes or no: did the mortgage run into payment trouble within 36 months (`trouble_36m` equal to 1) or not. This is classification, and the tool from the second supervised-learning class is logistic regression. It does not predict the outcome itself; it predicts a probability of trouble for each application, and the bank turns that probability into a decision by choosing a threshold above which it declines. The threshold is where the CRO's phrase "in francs" enters, and it is the subject of the second half of this part.

We follow the sequence of that class.

1. Look at the base rate, and at what a model that always predicts the majority class would score.
2. Pick the features, applying the decision-time rule column by column.
3. Split the book into a training and a test set, stratified so that both hold the same share of troubled loans, and standardise the features on the training set only.
4. Fit the logistic regression and read its coefficients as odds ratios.
5. Turn the test-set probabilities into decisions at a threshold and read the confusion matrix.
6. Judge the ranking independently of any threshold with the ROC curve and its AUC, computed from the probabilities.
7. Put a price on the two kinds of error, in CHF, and ask where the threshold should sit.

The decision-time rule bites here for the first time. The column dictionary marks one column of the book as known only afterwards: `reminders_sent`, the number of payment reminders the bank sent during the 36 months. It is the column that the applications file does not have, and the cell below shows why it cannot be a predictor: it is recorded after the outcome it would predict, and it is to a large extent the same event seen from the bank's side. Three further columns are known at decision time but are left out for the reasons the comment gives. `purchase_price` and `loan_amount` already enter through the three ratios, and `origination_year` takes a value in the applications, 2026, that the book never contains, so a coefficient estimated on 2019 to 2022 could not be applied to it.

In [ ]:
# Which columns does the bank know when it decides? Everything in the file except the outcome and one more.
# reminders_sent counts the payment reminders sent DURING the 36 months: it is a consequence of trouble,
# not a predictor. A model that uses it looks brilliant on the book and is useless for an application.
print(book.groupby("trouble_36m").reminders_sent.mean().rename("mean reminders_sent"))
print()
# purchase_price and loan_amount are known, but they enter through ltv / affordability / actual_burden;
# origination_year is known, but this year's applications come from 2026, a year the book never saw.
# overpayment, if Exercise 2 created it, is left for Part B, so that every team sees the same numbers here.
target = "trouble_36m"
not_features = ["id", "purchase_price", "loan_amount", "origination_year", "reminders_sent", "overpayment", target]
features = [c for c in book.columns if c not in not_features]
print("features used:", features)

In [ ]:
# Base rate and the majority-class baseline: a "model" that approves everyone is right 93% of the time
print(f"trouble rate: {book[target].mean():.3%}")
print(f"accuracy of 'nobody gets into trouble': {1 - book[target].mean():.3%}")

In [ ]:
# Split first (stratified on the outcome), then standardise on the training set only
X = pd.get_dummies(book[features], drop_first=True).astype(float)
y = book[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)
print("train:", X_train.shape, "| test:", X_test.shape, "| columns:", list(X.columns))

In [ ]:
# Logistic regression. Coefficients are log-odds per one standard deviation of the feature (we standardised),
# and exp(coef) is the odds ratio: how much the odds of trouble multiply when the feature rises by one SD.
logit = LogisticRegression(max_iter=2000).fit(X_train_s, y_train)
p_logit = logit.predict_proba(X_test_s)[:, 1]

odds = pd.DataFrame({"coef (log-odds per SD)": logit.coef_[0], "odds ratio": np.exp(logit.coef_[0])},
                    index=X.columns).sort_values("coef (log-odds per SD)", key=abs, ascending=False)
odds.round(3)

Every row of the table is an odds ratio, and an odds ratio is a multiplier on the odds of trouble when the feature rises by one standard deviation, holding the other columns fixed. In this run the largest is `affordability`: one standard deviation more, which is about 0.13 more of gross income needed to carry the imputed cost, multiplies the odds of trouble by 2.7. The Swiss rule looks at the right ratio. Next come `household_income` with 1.7 and `rate_type_saron` with 1.7, then `ltv` with 1.6, where one standard deviation is about ten points of loan-to-value.

Two of these need care. The income row points the way a reader may not expect: more income, higher odds of trouble. It is read holding the three ratios fixed, and at the same affordability and loan-to-value more income means a larger loan, so the row says that larger exposures at the same ratios go wrong more often, not that the bank should prefer poorer households. A coefficient on a column that also sits in the denominator of three other features is the clearest warning in the table against reading any row on its own. The SARON row is a 0/1 column, so one standard deviation is not a switch from a fixed to a variable rate; that switch is about two standard deviations, and the odds of trouble for a SARON mortgage are about 2.9 times those of a comparable fixed-rate one.

An odds ratio below 1 is protective. A property in the canton of Zurich has about 0.7 times the odds of trouble of a comparable one in Aargau, and one standard deviation more living area, about 38 m², multiplies the odds by 0.72. At the bottom of the table `rooms` (0.97), `years_client` (0.97) and the energy-label dummies (between 0.88 and 1.08, in no order from B to G) sit close to 1. A column with an odds ratio of 1 changes nothing, and this is the first hint that not every column in the file carries information about repayment. Part B returns to it with a method that selects features.

In [ ]:
# From probabilities to decisions: the 0.5 threshold, the confusion matrix and the two error rates
pred_05 = (p_logit >= 0.5).astype(int)
cm = confusion_matrix(y_test, pred_05)
print("confusion matrix (rows: actual 0/1, columns: predicted 0/1)\n", cm)
print(f"accuracy  : {accuracy_score(y_test, pred_05):.3%}   (majority baseline {1 - y_test.mean():.3%})")
print(f"recall    : {recall_score(y_test, pred_05):.3%}   share of troubled loans the model flags")
print(f"precision : {precision_score(y_test, pred_05, zero_division=0):.3%}   share of flagged loans that are troubled")

In [ ]:
# ROC curve and AUC, computed from the probabilities, never from the 0/1 predictions
fpr, tpr, thr = roc_curve(y_test, p_logit)
auc_logit = roc_auc_score(y_test, p_logit)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, lw=2, label=f"logit, AUC = {auc_logit:.3f}")
ax.plot([0, 1], [0, 1], "--", color="grey", label="coin flip, AUC = 0.5")
ax.set_xlabel("false positive rate (good loans flagged)"); ax.set_ylabel("true positive rate (troubled loans flagged)")
ax.set_title("ROC curve on the test set"); ax.legend()
plt.show()

In [ ]:
# The two errors have different prices. Expected cost per 1,000 applications at a given threshold:
def cost_per_1000(y_true, prob, threshold):
    pred = (np.asarray(prob) >= threshold).astype(int)
    fn = int(((pred == 0) & (np.asarray(y_true) == 1)).sum())   # approved, ran into trouble
    fp = int(((pred == 1) & (np.asarray(y_true) == 0)).sum())   # rejected, would have been fine
    return (fn * COST_FN + fp * COST_FP) / len(y_true) * 1000

def best_threshold(y_true, prob, grid=np.arange(0.02, 0.51, 0.01)):
    costs = [cost_per_1000(y_true, prob, t) for t in grid]
    return float(grid[int(np.argmin(costs))])

print(f"cost per 1,000 applications at threshold 0.5   : CHF {cost_per_1000(y_test, p_logit, 0.5):,.0f}")
print(f"cost per 1,000 applications approving everyone : CHF {cost_per_1000(y_test, p_logit, 1.01):,.0f}")

At the conventional threshold of 0.5 the model declines an application only when it thinks trouble is more likely than not, and in this run that is almost nobody: 48 of the 3,000 test applications, of which 28 did run into trouble and 20 would have been fine. The other 180 troubled loans pass. Recall is 13.5 %, precision 58.3 %, and accuracy is 93.3 % against 93.1 % for the baseline that approves everyone, which is the number the CRO must never be shown on its own. The ROC curve tells a different story: an AUC of 0.816 in this run means that the model's ranking of the applications carries real information, and the 0.5 threshold throws most of it away.

The costs say the same in francs. At 0.5 the expected cost is CHF 3,653,333 per 1,000 applications, against CHF 4,160,000 for approving everyone. Almost all of it, CHF 3.6 million, is the 180 approved loans that went into trouble; the 20 rejected good loans add CHF 53,333. The reason is the CRO's price list. A troubled loan costs CHF 60,000 and a rejected good one CHF 8,000, so a troubled loan costs 7.5 times a lost customer. Declining pays as soon as the expected loss from approving, the probability of trouble times CHF 60,000, exceeds the expected loss from declining, the probability that the loan would have been fine times CHF 8,000. That happens at a probability of trouble of 8,000 / 68,000, about 12 %. The bar belongs well below 0.5, and Exercise 3 finds it empirically by sweeping the threshold.

One deliberate difference from the logistic-regression class: there we undersampled the majority class of the training set so that the model saw a balanced world, and shifted the intercept back afterwards. Here we do not, because the threshold is chosen in CHF, and a cost calculation needs probabilities that mean what they say: a predicted 14 % must be a 14 % chance of trouble, which is what a fit on the book's own base rate gives. Undersampling inflates every probability, and the bar would then be a number on a scale nobody at the bank uses. The imbalance is handled where it belongs, at the threshold.

What the analyst tells the CRO: a logistic regression on the eighteen columns known at decision time ranks applications with an AUC of 0.816 on unseen loans, the affordability ratio, the loan-to-value ratio and a variable rate are the strongest signals, and at the textbook bar of 0.5 it would decline almost nobody and cost the bank nearly as much as approving everyone. Because a troubled loan costs 7.5 times a lost good customer, the approval bar belongs near a predicted probability of trouble of 12 % rather than 50 %, and Exercise 3 sets it in francs.

### Exercise 3: The approval bar in francs

Sweep the threshold from 0.02 to 0.50 in steps of 0.01 with `cost_per_1000`, plot the expected cost per 1,000 applications against the threshold, and report the cost-minimising threshold together with the recall, precision and cost at that threshold. (This sweep uses the test set; Part B does it properly on cross-validated training predictions.)

*Deliverable:* the threshold, the three numbers, and one sentence for the CRO on what the bar means: at which predicted probability of trouble does the bank stop approving?

In [ ]:
# Exercise 3: solution
grid = np.arange(0.02, 0.51, 0.01)
costs = [cost_per_1000(y_test, p_logit, t) for t in grid]
t_star = float(grid[int(np.argmin(costs))])
pred_star = (p_logit >= t_star).astype(int)
print(f"cost-minimising threshold: {t_star:.2f}")
print(f"cost per 1,000: CHF {min(costs):,.0f}  | recall {recall_score(y_test, pred_star):.3f}  | precision {precision_score(y_test, pred_star):.3f}")
print(f"share of applications rejected at that bar: {pred_star.mean():.1%}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(grid, costs, lw=2)
ax.axvline(t_star, color="C3", ls="--", label=f"minimum at {t_star:.2f}")
ax.set_xlabel("threshold on the predicted probability"); ax.set_ylabel("expected cost per 1,000 applications (CHF)")
ax.set_title("The approval bar that minimises expected cost"); ax.legend()
plt.show()

**What we expect them to find.** In this run the minimum lies at a threshold of 0.14, far below 0.5: the expected cost falls to CHF 2,638,667 per 1,000 applications, about CHF 1.0 million less than at 0.5 and about CHF 1.5 million less than approving everyone. At that bar the bank rejects 13.6 % of applications, recall is 0.553 and precision 0.283: it catches more than half of the troubled loans, and about seven in ten of the applications it declines would have been fine, which is the right trade-off when a troubled loan costs 7.5 times a rejected good one. The empirical bar sits close to the break-even probability of 12 % from the cost argument, and the curve is flat between about 0.10 and 0.14, so a team reporting 0.10 or 0.12 has the same story. The sentence for the CRO: "we stop approving once the model's probability of trouble exceeds 14 %".

### Exercise 4: The Swiss rule, checked on our own book

The affordability rule says the imputed cost may not exceed one third of gross income, and the standard maximum loan-to-value is 80 %. On the full book, compute the trouble rate for mortgages above and below the 1/3 affordability line, above and below 80 % LTV, and for the four combinations of the two.

*Deliverable:* a 2×2 table of trouble rates (affordability above/below 1/3 by LTV above/below 0.8) and one sentence on what it shows. Keep this table in mind for Part B.

In [ ]:
# Exercise 4: solution
over_aff = (book.affordability > 1 / 3).map({True: "affordability > 1/3", False: "affordability <= 1/3"})
over_ltv = (book.ltv > 0.80).map({True: "LTV > 80%", False: "LTV <= 80%"})
print(book.groupby(over_aff).trouble_36m.mean().round(3), "\n")
print(book.groupby(over_ltv).trouble_36m.mean().round(3), "\n")
table = book.pivot_table(index=over_aff, columns=over_ltv, values="trouble_36m", aggfunc=["mean", "count"])
table.round(3)

**What we expect them to find.** Each rule on its own separates modestly in this run: mortgages above the one-third affordability line have a trouble rate of 16.2 % against 3.2 % below it, and mortgages above 80 % LTV 24.7 % against 3.8 %. The 2×2 table shows that the two rules do not add up, they multiply. Breaching only the affordability rule gives 5.9 %, breaching only the LTV rule 4.2 %, both within the rules 3.1 %, and the 834 mortgages that breach both have a trouble rate of 41.1 %, seven to thirteen times the other three cells. That corner is an interaction, and a logistic regression with main effects, which can only add one log-odds term per rule, cannot represent it. This is the seed of Part B.